In [2]:
import fastf1
import os

os.makedirs('../data/raw/cache', exist_ok=True)
fastf1.Cache.enable_cache('../data/raw/cache')

session = fastf1.get_session(2023, 'Bahrain', 'Q')
session.load()
print(session.laps[['Driver', 'LapTime', 'Compound']].head(10))

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

  Driver                LapTime Compound
0    VER                    NaT     SOFT
1    VER                    NaT     SOFT
2    VER 0 days 00:01:31.295000     SOFT
3    VER 0 days 00:01:49.812000     SOFT
4    VER                    NaT     SOFT
5    VER 0 days 00:02:12.516000     SOFT
6    VER                    NaT     SOFT
7    VER 0 days 00:01:30.503000     SOFT
8    VER 0 days 00:02:05.756000     SOFT
9    VER                    NaT     SOFT


In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(open(Path('config.yaml')))


In [3]:
import pandas as pd
import os

SEASONS = cfg['data']['seasons']

def get_session_laps(year, gp, session_name):
    try:
        session = fastf1.get_session(year, gp, session_name)
        session.load(telemetry=False, weather=True, messages=False)
        laps = session.laps.pick_quicklaps().copy()

        # Attach weather
        weather = session.weather_data
        laps['AirTemp']   = weather['AirTemp'].mean()
        laps['TrackTemp'] = weather['TrackTemp'].mean()
        laps['Humidity']  = weather['Humidity'].mean()
        laps['Rainfall']  = weather['Rainfall'].any()

        laps['Year']       = year
        laps['GrandPrix']  = gp
        laps['Session']    = session_name
        return laps

    except Exception as e:
        print(f"Skipped: {year} {gp} {session_name} — {e}")
        return pd.DataFrame()

def build_dataset(seasons=SEASONS):
    all_data = []
    for year in seasons:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        for _, event in schedule.iterrows():
            gp = event['EventName']
            print(f"\n Loading {year} — {gp}")
            for sess in ['FP1', 'FP2', 'FP3', 'Q']:
                df = get_session_laps(year, gp, sess)
                if not df.empty:
                    all_data.append(df)

    return pd.concat(all_data, ignore_index=True)

# Run it
raw_df = build_dataset()
os.makedirs('../data/raw', exist_ok=True)
raw_df.to_csv('../data/raw/all_sessions.csv', index=False)
print(f"\n✅ Done! Saved {len(raw_df)} rows.")
